# Mars Surface and Air Temperature History + LSTM (Curiosity REMS)

**What this notebook does, in plain words**

1. Downloads NASA's REMS weather-station data from the Curiosity rover (public, no account needed).
2. Builds a **table of air temperature and ground temperature (in kelvin)** for each hour of each Martian day (sol), and saves it as **CSV and Excel**.
3. Trains a small **LSTM neural network** that looks at the previous 24 Martian hours and predicts the **next hour's air and ground temperature**, and compares it with a naive baseline.

**Important:** unlike the lunar Diviner notebook (which uses long-term averages), this one uses a **real, chronological history**: one weather station, measuring day after day since August 2012. That is what makes an LSTM meaningful here.

**Time units:** one sol (a Martian day) lasts about 24 h 39 min. Here, one "hour" means 1/24 of a sol (about 61.7 minutes) of *local mean solar time*.

**How to use it:** run every cell from top to bottom. The first run is a *quick test* (a few sols) to make sure the download works. Then set `QUICK_TEST = False` in Step 1 and run again for the full history and the LSTM.

## Step 1 — Choose the data range and model settings

`BASE_URL` is the `DATA` folder of the REMS **reduced data (RDR)** volume on the NASA PDS Atmospheres Node. It holds one subfolder per range of sols, and the notebook only opens the ones it needs. If the first run says the folder cannot be opened, go to the REMS page
(https://atmos.nmsu.edu/data_and_services/atmospheres_data/MARS/curiosity/rems.html), click the link under *REDUCED DATA - Volume Directory*, and paste that address into `BASE_URL`.

In [ ]:
BASE_URL = 'https://atmos.nmsu.edu/PDS/data/mslrem_1001/DATA/'   # folder with the RDR files, split into SOL_xxxxx_xxxxx subfolders
SOL_START = 1                   # first sol to download (sol 1 = 7 Aug 2012)
SOL_END = 400                   # last sol (a Martian year is about 668 sols; use 700 to cover a full year, heavier download)
QUICK_TEST = True               # True = only QUICK_SOLS sols (fast check, no real training). False = full run.
QUICK_SOLS = 12

LOOKBACK = 24                   # LSTM input: the previous 24 Martian hours
HORIZON = 1                     # predict this many hours ahead (1 = next hour)
EPOCHS = 40
BATCH_SIZE = 64
TEST_FRACTION = 0.2             # the LAST 20% of the history is kept for testing (no shuffling)
STEPS_PER_SOL = 24              # one step = 1/24 of a sol

SOLS = (range(SOL_START, SOL_START + QUICK_SOLS) if QUICK_TEST else range(SOL_START, SOL_END + 1))
print(f'Sols: {SOLS.start}-{SOLS.stop - 1} ({len(SOLS)} sols) | quick test: {QUICK_TEST}')

## Step 2 — Download tools (no need to read this cell)

These functions find the REMS files on the NASA server (one file per sol), read the column names from each file's label (`.LBL`), pick the air-temperature, ground-temperature and local-time columns automatically, and keep the valid readings.

In [ ]:
import io, os, re, time, math, threading
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

HEADERS = {'User-Agent': 'mars-rems-lstm/1.0 (research notebook)'}
CACHE = 'rems_cache'
os.makedirs(CACHE, exist_ok=True)
# Example file name: RME_417331939RMD02240000000_______P7.TAB  ->  RMD + 4-digit sol (0224)
FILE_RE = re.compile(r'RME_\d+RMD(\d{4})\d*_+P\d+\.TAB$', re.I)

def _get(url, timeout=120, tries=3):
    last = None
    for k in range(tries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            time.sleep(1.5 * (k + 1))
    raise last

def _list_dir(url):
    return [urljoin(url, h) for h in re.findall(r'href="([^"]+)"', _get(url, timeout=30, tries=2).text, flags=re.I)]

SOLDIR_RE = re.compile(r'SOL_(\d+)_(\d+)', re.I)
ONESOL_RE = re.compile(r'SOL(\d{5})', re.I)

def index_rmd_files(base, sol_min, sol_max, max_depth=4):
    # Walk the server folders and return {sol: url} for the REMS MOD RDR files (names containing 'RMD').
    found, visited, todo, unmatched = {}, set(), [(base, 0)], []
    while todo:
        url, depth = todo.pop()
        if url in visited or depth > max_depth:
            continue
        visited.add(url)
        print('  listing', url)
        for link in _list_dir(url):
            if '?' in link or not link.startswith(url) or link == url:   # skip sort links and 'parent directory'
                continue
            seg = link.split('?')[0].rstrip('/').split('/')[-1]
            if link.split('?')[0].endswith('/'):
                r = SOLDIR_RE.fullmatch(seg)                      # e.g. SOL_00001_00089 (a range of sols)
                if r and (int(r.group(2)) < sol_min or int(r.group(1)) > sol_max):
                    continue
                o = ONESOL_RE.fullmatch(seg)                      # e.g. SOL00052 (one sol)
                if o and not (sol_min <= int(o.group(1)) <= sol_max):
                    continue
                todo.append((link, depth + 1))
                continue
            name = seg
            if name.upper().endswith('.TAB') and 'RMD' in name.upper():
                p = re.search(r'/SOL(\d{5})/', link)             # the sol comes from the folder name
                if p:
                    sol = int(p.group(1))
                else:
                    q = re.search(r'RMD(\d{4})', name, re.I)
                    sol = int(q.group(1)) if q else None
                if sol is not None and sol_min <= sol <= sol_max:
                    if sol not in found or name > found[sol].split('/')[-1]:   # keep the latest version
                        found[sol] = link
            elif len(unmatched) < 15 and name.upper().endswith(('.TAB', '.LBL', '.CSV')):
                unmatched.append(name)
    if not found and unmatched:
        print('No file with RMD in its name was found. Files seen instead (send this list to get the code adjusted):')
        for n in unmatched:
            print('   ', n)
    return found

def parse_label(text):
    # Column names, in order, from a PDS3 label or format (.FMT) file. Vector columns (ITEMS = n) are expanded.
    names = []
    for m in re.finditer(r'OBJECT\s*=\s*COLUMN(.*?)END_OBJECT\s*=\s*COLUMN', text, flags=re.S | re.I):
        blk = m.group(1) + '\n'
        n = re.search(r'\bNAME\s*=\s*"?([^"\r\n]+?)"?\s*[\r\n]', blk)
        base = n.group(1).strip() if n else f'col{len(names)}'
        it = re.search(r'\bITEMS\s*=\s*(\d+)', blk)
        k = int(it.group(1)) if it else 1
        names += [base] if k == 1 else [f'{base}_{j + 1}' for j in range(k)]
    return names

_fmt_lock = threading.Lock()

def get_column_names(lbl_text, tab_url, ncols):
    # The label either lists the columns itself or points to a format file (^STRUCTURE = "XXX.FMT") in the volume's LABEL folder.
    candidates = [parse_label(lbl_text)]
    root = tab_url.split('/DATA/')[0] + '/'
    for m in re.finditer(r'\^STRUCTURE\s*=\s*"?([^"\r\n]+?)"?\s*[\r\n]', lbl_text + '\n', flags=re.I):
        fmt = m.group(1).strip()
        path = os.path.join(CACHE, fmt)
        with _fmt_lock:
            if not os.path.exists(path):
                for u in (root + 'LABEL/' + fmt, root + 'LABEL/' + fmt.upper(), tab_url.rsplit('/', 1)[0] + '/' + fmt):
                    try:
                        content = _get(u, timeout=30, tries=1).content
                        with open(path, 'wb') as f:
                            f.write(content)
                        break
                    except Exception:
                        continue
        if os.path.exists(path):
            candidates.append(parse_label(open(path, encoding='latin-1').read()))
    for names in candidates:
        if len(names) == ncols:
            return names
    return max(candidates, key=len)

def pick_column(cols, options, avoid=('UNC', 'CONF', 'LEVEL')):
    for must in options:
        for c in cols:
            cu = str(c).upper().replace(' ', '_')
            if all(k in cu for k in must) and not any(a in cu for a in avoid):
                return c
    return None

def parse_hour(value):
    # '0224M10:33:20.123' -> 10.55 (local mean solar time in hours)
    m = re.search(r'(\d{1,2}):(\d{2}):(\d{2})', str(value))
    return (int(m.group(1)) + int(m.group(2)) / 60 + int(m.group(3)) / 3600) if m else np.nan

_printed = {'cols': False}

def read_sol(sol, url):
    tab = os.path.join(CACHE, os.path.basename(url))
    lbl = tab[:-4] + '.LBL'
    if not os.path.exists(tab):
        with open(tab, 'wb') as f:
            f.write(_get(url).content)
    if not os.path.exists(lbl):
        for ext in ('.LBL', '.lbl'):
            try:
                with open(lbl, 'wb') as f:
                    f.write(_get(url[:-4] + ext).content)
                break
            except Exception:
                continue
    lbl_text = open(lbl, encoding='latin-1').read() if os.path.exists(lbl) else ''
    df = pd.read_csv(tab, header=None, skipinitialspace=True, quotechar='"', on_bad_lines='skip', encoding='latin-1')
    names = get_column_names(lbl_text, url, df.shape[1])
    if len(names) == df.shape[1]:
        df.columns = names
    else:
        raise ValueError(f'Sol {sol}: found {len(names)} column names but the table has {df.shape[1]} columns. '
                         f'Label starts with: {lbl_text[:300]!r}')
    air = pick_column(df.columns, [('AMBIENT', 'TEMP'), ('AIR', 'TEMP')])
    ground = pick_column(df.columns, [('BRIGHTNESS', 'TEMP'), ('GROUND', 'TEMP')])
    tcol = pick_column(df.columns, [('LMST',), ('LTST',), ('LOCAL', 'SOLAR', 'TIME')], avoid=())
    if air is None or ground is None:
        raise ValueError(f'Could not recognise temperature columns. Columns found: {list(df.columns)}')
    if not _printed['cols']:
        print(f'Columns used -> air: {air} | ground: {ground} | local time: {tcol}')
        _printed['cols'] = True
    out = pd.DataFrame({'sol': sol,
                        'air_K': pd.to_numeric(df[air], errors='coerce'),
                        'ground_K': pd.to_numeric(df[ground], errors='coerce')})
    if tcol is not None:
        out['hour'] = df[tcol].map(parse_hour)
    else:                                                   # fallback: position inside the file
        out['hour'] = np.arange(len(df)) / max(len(df), 1) * 24
    return out

def download_all(index, sols, workers=4):
    parts, failed = [], []
    todo = [(s, index[s]) for s in sols if s in index]
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(read_sol, s, u): s for s, u in todo}
        for i, fut in enumerate(as_completed(futs), 1):
            try:
                parts.append(fut.result())
            except Exception as e:
                failed.append((futs[fut], str(e)[:150]))
            if i % 25 == 0 or i == len(futs):
                print(f'  {i}/{len(futs)} sols processed ({time.time() - t0:.0f}s)')
    if failed:
        print(f'{len(failed)} sols failed; first errors: {failed[:3]}')
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

## Step 3 — Download the data

First it lists the files on the server, then it downloads one small file per sol. Files already downloaded are kept in the `rems_cache` folder, so if the connection drops you can run this cell again and it continues where it stopped. In Colab that folder is erased when the session ends.

In [ ]:
print('Looking up files on the NASA PDS Atmospheres server...')
index = index_rmd_files(BASE_URL, SOLS.start, SOLS.stop - 1)
print(f'{len(index)} sol files found in the requested range.')
if not index:
    raise RuntimeError('No REMS files found. Check BASE_URL (see the note in Step 1) and the sol range.')

raw = download_all(index, SOLS)
if raw.empty:
    raise RuntimeError('Nothing could be read. See the error messages above.')

# drop obviously invalid values (fill values, sensor glitches)
raw.loc[(raw['air_K'] < 100) | (raw['air_K'] > 350), 'air_K'] = np.nan
raw.loc[(raw['ground_K'] < 100) | (raw['ground_K'] > 350), 'ground_K'] = np.nan
raw = raw.dropna(subset=['hour'], how='any')
raw = raw.dropna(subset=['air_K', 'ground_K'], how='all')
print(f'Readings kept: {len(raw):,} across {raw["sol"].nunique()} sols')

## Step 4 — Build the table

For each sol and each of the 24 Martian hours, the table gives the **mean, minimum and maximum temperature (K)** of the air and of the ground, and how many readings fell in that hour. REMS does not measure continuously (it takes about 5 minutes every hour plus longer sessions at some times), so a few hours can be empty.

In [ ]:
raw['hour_bin'] = np.floor(raw['hour']).astype(int).clip(0, 23)
raw['step'] = raw['sol'] * STEPS_PER_SOL + raw['hour_bin']      # one number per Martian hour since landing

table = (raw.groupby(['sol', 'hour_bin', 'step'])
            .agg(n_readings=('air_K', 'size'),
                 air_mean_K=('air_K', 'mean'), air_min_K=('air_K', 'min'), air_max_K=('air_K', 'max'),
                 ground_mean_K=('ground_K', 'mean'), ground_min_K=('ground_K', 'min'), ground_max_K=('ground_K', 'max'))
            .reset_index().round(2))

summary = (table.groupby('sol')
                .agg(air_coldest_K=('air_min_K', 'min'), air_warmest_K=('air_max_K', 'max'),
                     ground_coldest_K=('ground_min_K', 'min'), ground_warmest_K=('ground_max_K', 'max'))
                .round(2).reset_index())

name = 'mars_rems_temperature' + ('_QUICKTEST' if QUICK_TEST else '')
table.to_csv(name + '.csv', index=False)
try:
    with pd.ExcelWriter(name + '.xlsx') as xw:
        table.to_excel(xw, sheet_name='hourly_table', index=False)
        summary.to_excel(xw, sheet_name='daily_summary', index=False)
    saved = [name + '.csv', name + '.xlsx']
except Exception as e:
    print('Excel file not written (', e, '); CSV is saved.')
    saved = [name + '.csv']
print('Saved:', saved)
if QUICK_TEST:
    print(f'QUICK TEST: only {len(SOLS)} sols. Set QUICK_TEST = False in Step 1 for the full history and the LSTM.')

try:
    display(summary.head(10)); display(table.head(24))
except NameError:
    print(summary.head(10).to_string(index=False)); print(table.head(24).to_string(index=False))

## Step 5 — Prepare the sequences for the LSTM

The hourly means of air and ground temperature are placed on a regular time grid. Small gaps (up to 3 hours) are filled by straight-line interpolation; windows that still contain gaps are discarded. Each example is: *the previous 24 hours of (air, ground)* → *air and ground `HORIZON` hours later*. The **last 20% of the history** is held out for testing, and the scaling uses only the training period, so the model never sees the future.

In [ ]:
grid = (table.set_index('step')[['air_mean_K', 'ground_mean_K']]
             .reindex(range(int(table['step'].min()), int(table['step'].max()) + 1)))
grid = grid.interpolate(limit=3, limit_area='inside')
vals = grid.to_numpy(dtype='float32')

ends = []                                   # index of the first target step of each window
for i in range(LOOKBACK, len(vals) - HORIZON + 1):
    if not (np.isnan(vals[i - LOOKBACK:i]).any() or np.isnan(vals[i + HORIZON - 1]).any()):
        ends.append(i)
print(f'Time grid: {len(vals)} hours | usable windows: {len(ends)}')

enough = len(ends) >= 100
if enough:
    split = int(len(ends) * (1 - TEST_FRACTION))
    cut = ends[split]                                       # test windows start here
    mu = np.nanmean(vals[:cut], axis=0)
    sd = np.nanstd(vals[:cut], axis=0)
    scaled = (vals - mu) / sd
    X = np.stack([scaled[i - LOOKBACK:i] for i in ends]).astype('float32')
    y = np.stack([scaled[i + HORIZON - 1] for i in ends]).astype('float32')
    X_train, y_train, X_test, y_test = X[:split], y[:split], X[split:], y[split:]
    print(f'Train windows: {len(X_train)} | test windows: {len(X_test)} | input shape {X_train.shape[1:]}')
else:
    print('Not enough continuous data for an LSTM yet (normal in the quick test). '
          'Set QUICK_TEST = False in Step 1 and run again.')

## Step 6 — Train the LSTM

A small network (one LSTM layer of 64 units plus an output layer with 2 values: air and ground). Training stops early if the validation error stops improving. This cell needs TensorFlow, which is already installed in Google Colab.

In [ ]:
if not enough:
    print('Skipped: not enough data (see Step 5).')
else:
    import tensorflow as tf
    from tensorflow import keras
    tf.keras.utils.set_random_seed(42)

    model = keras.Sequential([
        keras.layers.Input(shape=(LOOKBACK, 2)),
        keras.layers.LSTM(64),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(2),
    ])
    model.compile(optimizer='adam', loss='mse')
    stop = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
    history = model.fit(X_train, y_train, validation_split=0.1,     # validation = last 10% of the training period
                        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True, callbacks=[stop], verbose=2)
    print('Training finished.')

## Step 7 — Results

Errors are in kelvin on the **test period** (the most recent part of the history). The **baseline** simply repeats the last known temperature; if the LSTM is not clearly better than the baseline, it has not learned anything useful beyond persistence.

In [ ]:
if not enough:
    print('Skipped: not enough data (see Step 5).')
else:
    pred = model.predict(X_test, verbose=0) * sd + mu
    true = y_test * sd + mu
    base = X_test[:, -1, :] * sd + mu                      # naive: repeat the last value

    rows = []
    for k, label in enumerate(['air', 'ground']):
        rows.append({'variable': label,
                     'LSTM_MAE_K': np.mean(np.abs(pred[:, k] - true[:, k])),
                     'LSTM_RMSE_K': np.sqrt(np.mean((pred[:, k] - true[:, k]) ** 2)),
                     'baseline_MAE_K': np.mean(np.abs(base[:, k] - true[:, k]))})
    results = pd.DataFrame(rows).round(2)
    try:
        display(results)
    except NameError:
        print(results.to_string(index=False))

    n_show = min(24 * 6, len(true))                        # last ~6 sols of the test period
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    for k, (label, color) in enumerate([('Air', '#0077B6'), ('Ground', '#E85D04')]):
        axes[k].plot(true[-n_show:, k], color='black', lw=1.5, label='actual')
        axes[k].plot(pred[-n_show:, k], color=color, lw=1.5, label='LSTM prediction')
        axes[k].set_ylabel(f'{label} temperature (K)')
        axes[k].grid(alpha=0.2); axes[k].legend(loc='upper right')
    axes[1].set_xlabel(f'Martian hours in the test period (prediction {HORIZON} h ahead)')
    axes[0].set_title('Curiosity REMS: LSTM forecast vs actual (test period)')
    plt.tight_layout(); plt.show()

## Step 8 — Plot the history and download the files

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
d = summary.sort_values('sol')
ax.fill_between(d['sol'], d['air_coldest_K'], d['air_warmest_K'], color='#0077B6', alpha=0.25, label='air: daily min to max')
ax.fill_between(d['sol'], d['ground_coldest_K'], d['ground_warmest_K'], color='#E85D04', alpha=0.25, label='ground: daily min to max')
ax.set_xlabel('Sol (Martian day since landing)'); ax.set_ylabel('Temperature (K)')
ax.set_title('Gale Crater (Curiosity, REMS): daily temperature range'); ax.grid(alpha=0.2); ax.legend()
plt.tight_layout(); plt.show()

try:                                       # in Google Colab this downloads the files to your computer
    from google.colab import files
    for f in saved:
        files.download(f)
except ImportError:
    pass

## Notes

- **Data source:** Mars Science Laboratory (Curiosity) REMS, *MOD RDR* reduced data, NASA PDS Atmospheres Node (NMSU). Temperatures are in kelvin. The ground value is the *brightness temperature* from the ground-temperature sensor (an infrared thermopile); the air value is the estimated *ambient* air temperature.
- **Single location:** Curiosity has been driving through Gale Crater, so the series is "weather along the rover's path", not one fixed spot.
- **Gaps:** REMS has not measured every hour of every sol, and some periods have little or no data. The notebook never invents long gaps; it only bridges up to 3 hours.
- **Season:** a Martian year is about 668 sols. With `SOL_END = 400` the model has seen only part of a year; with 700 it sees every season at least once, which usually helps the test period.
- **Why compare with a baseline:** one hour ahead, "same as now" is already a decent forecast. The interesting question is whether the LSTM beats it, and by how much.
- **Automatic column detection:** the notebook reads the column names from each file's label and prints the ones it chose (Step 3). If REMS changes its format, that is the first place to check.
- **Not verified live:** the data-reading cells were written from NASA's published documentation of the file format; the first run (quick test) is where any mismatch with the live server would show up.